# LEVIR-CD · U-Net eğitimi

Colab'da **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seç. Sonra **1–4** numaralı hücreleri sırayla çalıştır. İlk prova geçmeden tam eğitimi açma.

Kod GitHub'dan, `train.zip` ve `val.zip` [resmî LEVIR-CD kaynağından](https://justchenhao.github.io/LEVIR/) gelir. `test` bu notebook'ta kullanılmaz. Model kayıtları kendi Google Drive'ında kalır.

## 1. GPU ve proje kodu
GPU görünmüyorsa hücre burada durur; veri indirmez.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/apeates/change-detection.git'
REPO_DIR = Path('/content/change-detection')
DATA_ROOT = Path('/content/levir_cd_data')
BATCH_SIZE = 2
LR_SCHEDULE = 'linear'
RUN_NAME = f'early_fusion_v2_{LR_SCHEDULE}'

if not shutil.which('nvidia-smi') or subprocess.run(['nvidia-smi', '-L'], check=False).returncode != 0:
    raise RuntimeError('Colab çalışma zamanı ayarından GPU seçip bu hücreyi tekrar çalıştır.')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    '--branch', 'main', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print(f'Git commit: {REVISION}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR / 'requirements-train.txt'), 'gdown>=5.2,<7'], check=True)
subprocess.run([sys.executable, '-c',
                "import torch; assert torch.cuda.is_available(), 'PyTorch GPU erişimi yok'; print(torch.cuda.get_device_name(0))"],
               check=True)
sys.path.insert(0, str(REPO_DIR))

## 2. Eğitim kayıtları için Drive'ı bağla
Colab izin isterse kendi Google hesabında onayla.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/change-detection-runs') / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Normalde None kalsın. Resmî indirme çalışmazsa iki ZIP'i içeren kendi Drive klasörünü yaz.
DRIVE_ARCHIVES = None  # Örnek: Path('/content/drive/MyDrive/levir-cd')
print(f'Eğitim kayıtları: {RUN_ROOT}')

## 3. Veriyi hazırla
Eğitim için **445 train** ve **64 val** görüntü çifti gerekir. Sadece 6. hücredeki değerlendirmeyi yapacaksan `PREPARE_TRAIN = False` seç. Bu durumda yalnızca **64 val** görüntü çifti hazırlanır.

In [ ]:
import tempfile
import zipfile
from pathlib import PurePosixPath

import gdown
from levir_data import list_samples

PREPARE_TRAIN = True  # Yalnızca değerlendirme için False seç.
SPLITS = {
    'train': ('18GuoCuBn48oZKAlEo-LrNwABrFhVALU-', 445),
    'val': ('1BqSt4ueO7XAyQ_84mUjswUSJt13ZBuzG', 64),
}
ARCHIVE_CACHE = Path('/content/levir_cd_archives')

def verify_split(directory, expected):
    count = len(list_samples(directory))
    if count != expected:
        raise ValueError(f'{directory.name}: {count} eşleşme bulundu, {expected} bekleniyor. ZIP adını kontrol et.')
    return count

def get_archive(split, file_id):
    if DRIVE_ARCHIVES is not None:
        archive = Path(DRIVE_ARCHIVES) / f'{split}.zip'
    else:
        ARCHIVE_CACHE.mkdir(parents=True, exist_ok=True)
        archive = ARCHIVE_CACHE / f'{split}.zip'
        if not zipfile.is_zipfile(archive):
            downloaded = gdown.download(id=file_id, output=str(archive), quiet=False)
            if not downloaded:
                raise RuntimeError(f'{split}.zip indirilemedi. Hata mesajını kontrol et veya DRIVE_ARCHIVES kullan.')
    if not zipfile.is_zipfile(archive):
        raise ValueError(f'Geçerli ZIP bulunamadı: {archive}')
    return archive

def extract_split(archive, destination, expected):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=destination.parent) as directory:
        staged = Path(directory) / destination.name
        for folder in ('A', 'B', 'label'):
            (staged / folder).mkdir(parents=True)
        with zipfile.ZipFile(archive) as source:
            for entry in source.infolist():
                member = PurePosixPath(entry.filename)
                folder = member.parent.name
                if entry.is_dir() or member.suffix.lower() != '.png' or member.name.startswith('.'):
                    continue
                if folder not in ('A', 'B', 'label'):
                    continue
                target = staged / folder / member.name
                with source.open(entry) as incoming, target.open('xb') as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        verify_split(staged, expected)
        staged.replace(destination)

for split, (file_id, expected) in SPLITS.items():
    if split == 'train' and not PREPARE_TRAIN:
        continue
    destination = DATA_ROOT / split
    if not destination.exists():
        extract_split(get_archive(split, file_id), destination, expected)
    print(f'{split}: {verify_split(destination, expected)} eşleşen görüntü çifti')

## 4. Kısa GPU provası
Dört eğitim parçası, bir doğrulama görüntüsü, iki tur. Bu aşamadaki skor model başarısı olarak kullanılmaz. Çıktıda `Last:` ve `Best:` yollarını görmelisin.

In [ ]:
import torch

def run_training(name, epochs, smoke_tiles=None):
    output = RUN_ROOT / name
    output.mkdir(parents=True, exist_ok=True)
    revision_file = output / 'git_revision.txt'
    if revision_file.exists() and revision_file.read_text().strip() != REVISION:
        raise RuntimeError('Bu kayıt başka bir Git commit’inden. Yeni bir RUN_NAME seç.')
    revision_file.write_text(REVISION + '\n')
    command = [sys.executable, '-u', 'train_unet.py', '--device', 'cuda',
               '--data-root', str(DATA_ROOT), '--output-dir', str(output),
               '--epochs', str(epochs), '--batch-size', str(BATCH_SIZE),
               '--lr-schedule', LR_SCHEDULE]
    if smoke_tiles is not None:
        command.extend(['--smoke-tiles', str(smoke_tiles)])
    last = output / 'last.pt'
    if last.is_file():
        saved = torch.load(last, map_location='cpu', weights_only=True)
        if saved['epoch'] >= epochs:
            print(f'Bu koşu {saved["epoch"]} turu zaten tamamladı: {output}')
            return
        command.append('--resume')
    subprocess.run(command, cwd=REPO_DIR, check=True, env={**os.environ, 'MPLBACKEND': 'Agg'})

run_training('smoke', epochs=2, smoke_tiles=4)

## 5. Tam eğitim — prova başarılı olduktan sonra
İlk çalıştırmada bu hücreyi kapalı bırak. Prova başarılıysa `RUN_FULL = True` yap. Bu koşuda öğrenme hızı 10 turda `0.001`'den `0.0001`'e iner. Eğitim kesilirse aynı koşu adı, kod sürümü ve `FULL_EPOCHS` ile **1–4** hücrelerini tekrar çalıştır, sonra bu hücreyi aç. `last.pt` varsa son tamamlanan turdan devam eder.

In [ ]:
RUN_FULL = False
FULL_EPOCHS = 10

if RUN_FULL:
    smoke = RUN_ROOT / 'smoke'
    if not all((smoke / name).is_file() for name in ('last.pt', 'best.pt', 'git_revision.txt')):
        raise RuntimeError('Önce 4. hücredeki GPU provasını tamamla.')
    if (smoke / 'git_revision.txt').read_text().strip() != REVISION:
        raise RuntimeError('Bu kod sürümüyle prova yapmak için yeni bir RUN_NAME seç.')
    if torch.load(smoke / 'last.pt', map_location='cpu', weights_only=True)['epoch'] < 2:
        raise RuntimeError('Kısa prova henüz iki turu tamamlamadı; 4. hücreyi tekrar çalıştır.')
    run_training('full', epochs=FULL_EPOCHS)
else:
    print('Tam eğitim kapalı. Önce kısa GPU provasının çıktısını kontrol et.')

## 6. Eşikleri karşılaştır ve doğrulama hatalarını incele
İlk üç hücreyi çalıştır. 3. hücrede `PREPARE_TRAIN = False` seçebilirsin. Sonra doğrudan bu hücreye gel. Drive'daki `full/best.pt` ile yedi eşik tek geçişte ölçülür. Görseller ve görüntü başına CSV, önceki ölçümü karşılaştırabilmek için `0.35` eşiğini kullanır.

In [ ]:
import csv
import json
from IPython.display import Image, display

checkpoint = RUN_ROOT / 'full' / 'best.pt'
analysis_dir = RUN_ROOT / 'val_sweep_t035'
if not checkpoint.is_file():
    raise FileNotFoundError(f'Tamamlanan koşunun model kaydı bulunamadı: {checkpoint}')
subprocess.run([sys.executable, '-u', 'evaluate_unet.py', '--checkpoint', str(checkpoint),
                '--data-root', str(DATA_ROOT), '--threshold', '0.35', '--sweep',
                '--device', 'cuda', '--batch-size', str(BATCH_SIZE),
                '--output-dir', str(analysis_dir)], cwd=REPO_DIR, check=True,
               env={**os.environ, 'MPLBACKEND': 'Agg'})
summary = json.loads((analysis_dir / 'summary.json').read_text())
with (analysis_dir / 'threshold_sweep.csv').open(newline='') as file:
    print('Eşik  Precision   Recall       F1      IoU')
    for row in csv.DictReader(file):
        scores = ' '.join(f'{float(row[key]):8.2%}' for key in ('Precision', 'Recall', 'F1', 'IoU'))
        print(f"{float(row['threshold']):.2f}  {scores}")
print(f"En iyi val eşiği: {summary['threshold_sweep']['best_threshold']:.2f}")
print('0.35 sayımları:', summary['counts'])
print(f'Drive çıktıları: {analysis_dir}')
for example in summary['examples'].values():
    display(Image(filename=str(analysis_dir / example['figure']), width=1000))

## Sonuçlar nerede?
Drive: `change-detection-runs/early_fusion_v2_linear/`. `smoke/` ve `full/` ayrı durur. `history.csv` tur sonuçlarını ve öğrenme hızını, `last.pt` devam kaydını, `best.pt` en iyi `val` F1 kaydını tutar. İlk koşunun `early_fusion_v1/` kayıtları ayrı kalır.

6. hücre `val_sweep_t035/` altında eşik tablosunu, görüntü başına sonuçları ve hata görsellerini kaydeder. `test` ile son kıyası daha sonra yapacağız.